In [5]:
import os
import kagglehub

# 1. INGESTION & STANDARDIZATION (Fresh start)
# Downloading directly from your Kaggle cache like you did on Day 1
path = kagglehub.dataset_download("gustavoserafim/walmart-recruiting-store-sales-forecasting-gsr")
df = pd.read_csv(os.path.join(path, "train.csv"))

df.columns = df.columns.str.strip()
import pandas as pd
import numpy as np
import subprocess

print("🔄 Resetting Kernel State & Rebuilding Matrix...")


# Safely catch any capitalization differences in the raw file
df.rename(columns=lambda x: 'Date' if x.lower() == 'date' else x, inplace=True)
df.rename(columns=lambda x: 'IsHoliday' if x.lower() == 'isholiday' else x, inplace=True)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)

# 2. FEATURE ENGINEERING
grouped = df.groupby(['Store', 'Dept'])['Weekly_Sales']
df['Weekly_Sales_Lag_1'] = grouped.shift(1)
df['Weekly_Sales_Lag_4'] = grouped.shift(4)
df['Weekly_Sales_Lag_52'] = grouped.shift(52)

df['Rolling_Mean_4'] = grouped.transform(lambda x: x.shift(1).rolling(window=4).mean())
df['Rolling_Mean_12'] = grouped.transform(lambda x: x.shift(1).rolling(window=12).mean())
df['Rolling_Std_4'] = grouped.transform(lambda x: x.shift(1).rolling(window=4).std())

df['Month'] = df['Date'].dt.month
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsHoliday_NextWeek'] = df.groupby(['Store', 'Dept'])['IsHoliday'].shift(-1).fillna(False).astype(int)
df['IsHoliday_LastWeek'] = df.groupby(['Store', 'Dept'])['IsHoliday'].shift(1).fillna(False).astype(int)
df['IsHoliday'] = df['IsHoliday'].astype(int)

# 3. TRIMMING & VALIDATION SPLIT
df_clean = df.dropna().copy().reset_index(drop=True)
split_date = pd.to_datetime('2012-06-01')
train_set = df_clean[df_clean['Date'] < split_date]
test_set = df_clean[df_clean['Date'] >= split_date]

features = [
    'Store', 'Dept', 'IsHoliday',
    'Weekly_Sales_Lag_1', 'Weekly_Sales_Lag_4', 'Weekly_Sales_Lag_52',
    'Rolling_Mean_4', 'Rolling_Mean_12', 'Rolling_Std_4',
    'Month', 'Week_of_Year', 'IsHoliday_NextWeek', 'IsHoliday_LastWeek'
]

# 🔍 Diagnostics Check: Prints an empty set if everything worked!
missing_cols = set(features) - set(train_set.columns)
if missing_cols:
    print(f"⚠️ FATAL ERROR: Missing Columns: {missing_cols}")
else:
    print("✅ Matrix perfectly aligned! Slicing tensors...")

# 4. SLICING
X_train, y_train = train_set[features], train_set['Weekly_Sales']
X_test, y_test = test_set[features], test_set['Weekly_Sales']

# 5. LIGHTGBM TRAINING
%pip install lightgbm
try:
    import lightgbm as lgb
except ImportError:
    subprocess.check_call(["pip", "install", "lightgbm"])
    import lightgbm as lgb

print("⚡ Training LightGBM...")
model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
model.fit(X_train, y_train)

# 6. METRICS EVALUATION
preds = model.predict(X_test)
rmse = np.sqrt(np.mean((y_test - preds) ** 2))
print(f"\n🎯 FINAL RMSE: ${rmse:,.2f}")

🔄 Resetting Kernel State & Rebuilding Matrix...
✅ Matrix perfectly aligned! Slicing tensors...


C:\Users\Saurya prakash\AppData\Local\Temp\ipykernel_25104\3197849569.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['IsHoliday_NextWeek'] = df.groupby(['Store', 'Dept'])['IsHoliday'].shift(-1).fillna(False).astype(int)
C:\Users\Saurya prakash\AppData\Local\Temp\ipykernel_25104\3197849569.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['IsHoliday_LastWeek'] = df.groupby(['Store', 'Dept'])['IsHoliday'].shift(1).fillna(False).astype(int)



  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
⚡ Training LightGBM...


c:\Users\Saurya prakash\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Saurya prakash\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Saurya prakash\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Saurya prakash\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~


🎯 FINAL RMSE: $2,697.70
